In [51]:
"""
need to pip install a few modules:
    pandas
    scikit-learn
    matplotlib

"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.multiclass import OneVsRestClassifier

In [32]:
data_df = pd.read_csv("train.csv")

# three possible classifications: GALAXY, STAR, QSO
total_len = len(data_df)

In [33]:
data_df.head(10)


,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY
5,5,250.727869,31.756548,20.926469,19.693480,18.902361,19.247572,18.508241,0.076299,G/K,Blue_Cloud,STAR
6,6,0.752529,-2.936677,22.829195,22.686143,20.583886,19.781338,19.410491,0.575080,M,Red_Sequence,GALAXY
7,7,235.611325,39.626517,22.511467,21.480306,21.765645,21.508658,21.333476,2.159269,O/B,Blue_Cloud,QSO
8,8,355.359230,2.182312,20.396550,20.064767,19.892257,19.836272,19.860081,0.900087,A/F,Blue_Cloud,QSO
9,9,254.980080,38.743449,18.839137,17.997845,18.458894,18.229552,19.202247,0.114302,O/B,Blue_Cloud,STAR


In [34]:
test_df = pd.read_csv("test.csv")
test_df.head(10)

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence
5,577352,337.489471,32.273968,23.957231,23.252978,21.426614,20.642866,19.582499,0.998260,M,Red_Sequence
6,577353,218.849735,29.668026,21.266555,18.873621,17.714544,17.188945,16.769579,0.419590,M,Red_Sequence
7,577354,1.783080,7.749121,19.996539,18.476004,17.609485,17.723595,17.905090,0.060183,G/K,Red_Sequence
8,577355,242.077161,44.256061,25.130018,22.366451,20.390028,19.915563,18.972140,0.209312,M,Red_Sequence
9,577356,330.969636,0.095475,19.916332,18.833580,17.663625,17.018369,16.917590,0.433929,M,Red_Sequence


In [41]:
spectral = pd.get_dummies(data_df['spectral_type'], dtype=int)

galaxy_pop = pd.get_dummies(data_df['galaxy_population'], dtype=int)

In [42]:
new_df = data_df.drop(columns = ['spectral_type', 'galaxy_population'])
new_df = pd.concat([new_df, spectral, galaxy_pop], axis=1)

class_mapping = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}

In [45]:
X_data = new_df.drop(columns = ['id', 'class'])
X_data = X_data.apply(pd.to_numeric)
y_data = data_df['class'].map(class_mapping)

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.25, random_state = 12345)


In [44]:
print(new_df.head())

   id       alpha      delta          u          g          r          i  \
0   0  147.734256  16.959273  25.472123  21.895559  20.357926  19.257113   
1   1  127.988677  32.346716  20.778509  19.087062  17.587208  17.226067   
2   2  179.792648  35.344843  21.035203  21.079128  21.171840  20.582629   
3   3  225.818295  48.569421  23.305056  21.050736  19.017754  18.365658   
4   4  141.836135  19.342852  21.703158  19.471680  18.234449  17.899447   

           z  redshift   class  A/F  G/K  M  O/B  Blue_Cloud  Red_Sequence  
0  18.621057  0.408982  GALAXY    0    0  1    0           0             1  
1  16.786433  0.157976  GALAXY    0    0  1    0           0             1  
2  20.557366  2.823770     QSO    0    0  0    1           1             0  
3  17.914952  0.536099  GALAXY    0    0  1    0           0             1  
4  17.616185  0.555761  GALAXY    0    0  1    0           0             1  


In [49]:
print("Length of training data split: ", len(X_train), "and", len(y_train))
print("Length of testing section: ", len(X_test), "and", len(y_test))

Length of training data split:  433010 and 433010
Length of testing section:  144337 and 144337


In [52]:

# Wrap your model to explicitly enforce One-Vs-Rest
first_model = OneVsRestClassifier(LogisticRegression(solver='liblinear'))
first_model.fit(X_train, y_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...r='liblinear')
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
Name,Type,Value
"classes_ classes_: array, shape = [`n_classes`]Class labels.","ndarray[int64](3,)","[0,1,2]"
estimators_ estimators_: list of `n_classes` estimatorsEstimators used for predictions.,list,"[LogisticRegre...r='liblinear'), LogisticRegre...r='liblinear'), LogisticRegre...r='liblinear')]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['alpha','delta','u',...,'O/B','Blue_Cloud','Red_Sequence']"
label_binarizer_ label_binarizer_: LabelBinarizer objectObject used to transform multiclass labels to binary labels andvice-versa.,LabelBinarizer,LabelBinarize...e_output=True)
multilabel_ multilabel_: booleanWhether a OneVsRestClassifier is a multilabel classifier.,bool,False
n_classes_ n_classes_: intNumber of classes.,int,3
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,14


In [56]:
# Extract weights from all classes in the model
weights = np.array([estimator.coef_[0] for estimator in first_model.estimators_])

# Extract biases (intercepts) for all classes
biases = np.array([estimator.intercept_[0] for estimator in first_model.estimators_])

for target_class, w, b in zip(first_model.classes_, weights, biases):
    print(f"Class {target_class} -> Bias: {b:.4f} | Weights: {w}")

Class 0 -> Bias: 0.5822 | Weights: [-1.20065575e-03 -7.26509775e-03 -1.02059936e-01  1.85289991e+00
  1.12831799e+00 -1.36197072e+00 -1.66661848e+00 -5.02869988e-01
 -2.41841000e-02  4.24709811e-01  4.49768186e-01 -2.68076901e-01
  8.17602259e-02  5.00456770e-01]
Class 1 -> Bias: -1.3644 | Weights: [ 1.98115232e-03  2.33751471e-02 -1.77940087e+00 -5.45736138e-01
 -7.19201450e-01  1.92259052e+00  1.22870767e+00  3.39998569e+00
 -4.22819704e-02 -5.30278996e-01 -8.96437800e-01  1.04630335e-01
 -1.57367562e+00  2.09307190e-01]
Class 2 -> Bias: -1.6506 | Weights: [-3.49389389e-04 -7.20868066e-03  4.28845548e-01 -1.57427274e+00
  1.81003303e-01  2.81879335e-01  9.46722738e-01 -1.23186365e+01
 -2.24640908e-01 -2.11942988e-01 -4.07774241e-01 -8.06276398e-01
 -2.45144590e-01 -1.40548995e+00]
